# **Remaining Useful Life Estimation with Machine Learning**

## **Inits**

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning model
from sklearn.ensemble import RandomForestRegressor

# Model evaluation metrics
from sklearn.metrics import mean_squared_error, r2_score

# Data preprocessing and dataset splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Utility libraries
import os
import random
import warnings

# ============================================================
# Configuration and Environment Setup
# ============================================================

# Set NumPy random seed for reproducibility
np.random.seed(34)

# Suppress warning messages for cleaner output
warnings.filterwarnings('ignore')

## **Defining Train, Test**

In [2]:
# ============================================================
# Define Dataset Column Names
# ============================================================

# Engine identification and operational cycle columns
index_names = ['engine', 'cycle']

# Operational setting parameters
setting_names = ['setting_1', 'setting_2', 'setting_3']

# Sensor measurement names
sensor_names = [
    "(Fan inlet temperature) (◦R)",
    "(LPC outlet temperature) (◦R)",
    "(HPC outlet temperature) (◦R)",
    "(LPT outlet temperature) (◦R)",
    "(Fan inlet Pressure) (psia)",
    "(bypass-duct pressure) (psia)",
    "(HPC outlet pressure) (psia)",
    "(Physical fan speed) (rpm)",
    "(Physical core speed) (rpm)",
    "(Engine pressure ratio(P50/P2)",
    "(HPC outlet Static pressure) (psia)",
    "(Ratio of fuel flow to Ps30) (pps/psia)",
    "(Corrected fan speed) (rpm)",
    "(Corrected core speed) (rpm)",
    "(Bypass Ratio)",
    "(Burner fuel-air ratio)",
    "(Bleed Enthalpy)",
    "(Required fan speed)",
    "(Required fan conversion speed)",
    "(High-pressure turbines Cool air flow)",
    "(Low-pressure turbines Cool air flow)"
]

# Combine all column groups into a single list
col_names = index_names + setting_names + sensor_names


# ============================================================
# Load NASA C-MAPSS Dataset
# ============================================================

# Training dataset
df_train = pd.read_csv(
    '/kaggle/input/nasa-cmaps/CMaps/train_FD001.txt',
    sep=r'\s+',
    header=None,
    names=col_names
)

# Testing dataset
df_test = pd.read_csv(
    '/kaggle/input/nasa-cmaps/CMaps/test_FD001.txt',
    sep=r'\s+',
    header=None,
    names=col_names
)

# Remaining Useful Life (RUL) labels for test dataset
df_test_RUL = pd.read_csv(
    '/kaggle/input/nasa-cmaps/CMaps/RUL_FD001.txt',
    sep=r'\s+',
    header=None,
    names=['RUL']
)

In [3]:
df_train.head()

,engine,cycle,setting_1,setting_2,setting_3,(Fan inlet temperature) (◦R),(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(Fan inlet Pressure) (psia),...,(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Burner fuel-air ratio),(Bleed Enthalpy),(Required fan speed),(Required fan conversion speed),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow)
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [4]:
df_test.head()

,engine,cycle,setting_1,setting_2,setting_3,(Fan inlet temperature) (◦R),(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(Fan inlet Pressure) (psia),...,(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Burner fuel-air ratio),(Bleed Enthalpy),(Required fan speed),(Required fan conversion speed),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow)
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [5]:
df_test_RUL.head()

,RUL
0,112
1,98
2,69
3,82
4,91


In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   engine                                   20631 non-null  int64  
 1   cycle                                    20631 non-null  int64  
 2   setting_1                                20631 non-null  float64
 3   setting_2                                20631 non-null  float64
 4   setting_3                                20631 non-null  float64
 5   (Fan inlet temperature) (◦R)             20631 non-null  float64
 6   (LPC outlet temperature) (◦R)            20631 non-null  float64
 7   (HPC outlet temperature) (◦R)            20631 non-null  float64
 8   (LPT outlet temperature) (◦R)            20631 non-null  float64
 9   (Fan inlet Pressure) (psia)              20631 non-null  float64
 10  (bypass-duct pressure) (psia)            20631

## **Describe**

In [7]:
df_train.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
engine,20631.0,51.506568,2.922763e+01,1.0000,26.0000,52.0000,77.0000,100.0000
cycle,20631.0,108.807862,6.888099e+01,1.0000,52.0000,104.0000,156.0000,362.0000
setting_1,20631.0,-0.000009,2.187313e-03,-0.0087,-0.0015,0.0000,0.0015,0.0087
setting_2,20631.0,0.000002,2.930621e-04,-0.0006,-0.0002,0.0000,0.0003,0.0006
setting_3,20631.0,100.000000,0.000000e+00,100.0000,100.0000,100.0000,100.0000,100.0000
(Fan inlet temperature) (◦R),20631.0,518.670000,6.537152e-11,518.6700,518.6700,518.6700,518.6700,518.6700
(LPC outlet temperature) (◦R),20631.0,642.680934,5.000533e-01,641.2100,642.3250,642.6400,643.0000,644.5300
(HPC outlet temperature) (◦R),20631.0,1590.523119,6.131150e+00,1571.0400,1586.2600,1590.1000,1594.3800,1616.9100
(LPT outlet temperature) (◦R),20631.0,1408.933782,9.000605e+00,1382.2500,1402.3600,1408.0400,1414.5550,1441.4900
(Fan inlet Pressure) (psia),20631.0,14.620000,3.394700e-12,14.6200,14.6200,14.6200,14.6200,14.6200


## **Remocing Constant Columns**

In [8]:
# ============================================================
# Identify and Remove Constant Columns
# ============================================================

# Find columns that contain only a single unique value
# These columns do not provide useful information for modeling
constant_cols = [
    col for col in df_train.columns
    if df_train[col].nunique() == 1
]

# Display constant columns
print("Columns with constant values:", constant_cols)

# Remove constant columns from training dataset
df_train.drop(
    constant_cols,
    axis=1,
    inplace=True,
    errors='ignore'
)

# Remove the same constant columns from testing dataset
df_test.drop(
    constant_cols,
    axis=1,
    inplace=True,
    errors='ignore'
)

Columns with constant values: ['setting_3', '(Fan inlet temperature) (◦R)', '(Fan inlet Pressure) (psia)', '(Engine pressure ratio(P50/P2)', '(Burner fuel-air ratio)', '(Required fan speed)', '(Required fan conversion speed)']


## **Remaining Engine Life**

In [9]:
# ============================================================
# Calculate Remaining Engine Life Information
# ============================================================

# Group the training data by engine ID
# and find the maximum cycle value for each engine
# The maximum cycle represents the total operational life
# of each engine before failure
df_train_RUL = df_train.groupby(['engine']).agg({
    'cycle': 'max'
})

# Rename the 'cycle' column to 'life'
# for better interpretability
df_train_RUL.rename(
    columns={'cycle': 'life'},
    inplace=True
)

# Display the first few rows
df_train_RUL.head()

,life
engine,
1,192
2,287
3,179
4,189
5,269


In [10]:
df_train=df_train.merge(df_train_RUL,how='left',on=['engine'])

## **Remaining Useful Life (RUL**)

In [11]:
# ============================================================
# Calculate Remaining Useful Life (RUL)
# ============================================================

# RUL is calculated as:
# Remaining Useful Life = Total Engine Life - Current Cycle
df_train['RUL'] = df_train['life'] - df_train['cycle']


# ============================================================
# Remove Temporary Columns
# ============================================================

# The 'life' column is no longer needed after calculating RUL
df_train.drop(
    ['life'],
    axis=1,
    inplace=True
)


# ============================================================
# Apply RUL Capping
# ============================================================

# Limit maximum RUL values to 125
# This helps reduce the effect of extremely large target values
# and stabilizes model training
df_train.loc[df_train['RUL'] > 125, 'RUL'] = 125


# Display the first few rows of the updated dataset
df_train.head()

,engine,cycle,setting_1,setting_2,(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(bypass-duct pressure) (psia),(HPC outlet pressure) (psia),(Physical fan speed) (rpm),(Physical core speed) (rpm),(HPC outlet Static pressure) (psia),(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Bleed Enthalpy),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow),RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,125
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,125
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,125
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,125


## **Feature Selection using Backward Stepwise Regression**

In [12]:
# ============================================================
# Feature Selection using Backward Stepwise Regression
# ============================================================

# List to store selected features
Selected_Features = []

# Statistical modeling library
import statsmodels.api as sm


# ============================================================
# Define Backward Regression Function
# ============================================================

def backward_regression(
    X,
    y,
    initial_list=[],
    threshold_out=0.05,
    verbose=True
):
    """
    Perform Backward Stepwise Regression for feature selection.

    Parameters
    ----------
    X : DataFrame
        Input feature matrix.

    y : Series or array
        Target variable.

    initial_list : list, optional
        Initial list of selected features.
        (Not used directly here, included for flexibility.)

    threshold_out : float, optional
        Maximum allowed p-value.
        Features with p-values above this threshold
        will be removed iteratively.

    verbose : bool, optional
        If True, prints details of removed features.

    Returns
    -------
    list
        Final list of selected features.
    """

    # Start with all available features
    included = list(X.columns)

    while True:

        changed = False

        # Fit Ordinary Least Squares (OLS) regression model
        model = sm.OLS(
            y,
            sm.add_constant(pd.DataFrame(X[included]))
        ).fit()

        # Exclude intercept p-value and evaluate feature p-values only
        pvalues = model.pvalues.iloc[1:]

        # Find the highest p-value among features
        worst_pval = pvalues.max()

        # Remove feature if its p-value exceeds threshold
        if worst_pval > threshold_out:

            changed = True

            # Identify feature with highest p-value
            worst_feature = pvalues.idxmax()

            # Remove the least significant feature
            included.remove(worst_feature)

            # Print removed feature information
            if verbose:
                print(
                    f"Removed Feature: {worst_feature} "
                    f"| p-value: {worst_pval}"
                )

        # Stop when no more features need removal
        if not changed:
            break

    # Store selected features
    Selected_Features.append(included)

    # Display final selected features
    print("\nSelected Features:")
    print(Selected_Features[0])


# ============================================================
# Prepare Training Data for Feature Selection
# ============================================================

# Select feature columns
# Exclude:
#   - first column ('engine')
#   - last column ('RUL')
X = df_train.iloc[:, 1:-1]

# Target variable
y = df_train.iloc[:, -1]


# ============================================================
# Apply Backward Regression
# ============================================================

backward_regression(X, y)

worst_feature : setting_1, 0.358952959016297 
worst_feature : setting_2, 0.18229853655124673 

Selected Features:
['cycle', '(LPC outlet temperature) (◦R)', '(HPC outlet temperature) (◦R)', '(LPT outlet temperature) (◦R)', '(bypass-duct pressure) (psia)', '(HPC outlet pressure) (psia)', '(Physical fan speed) (rpm)', '(Physical core speed) (rpm)', '(HPC outlet Static pressure) (psia)', '(Ratio of fuel flow to Ps30) (pps/psia)', '(Corrected fan speed) (rpm)', '(Corrected core speed) (rpm)', '(Bypass Ratio) ', '(Bleed Enthalpy)', '(High-pressure turbines Cool air flow)', '(Low-pressure turbines Cool air flow)']


### **Selected Features**

In [13]:
Selected_Features

[['cycle',
  '(LPC outlet temperature) (◦R)',
  '(HPC outlet temperature) (◦R)',
  '(LPT outlet temperature) (◦R)',
  '(bypass-duct pressure) (psia)',
  '(HPC outlet pressure) (psia)',
  '(Physical fan speed) (rpm)',
  '(Physical core speed) (rpm)',
  '(HPC outlet Static pressure) (psia)',
  '(Ratio of fuel flow to Ps30) (pps/psia)',
  '(Corrected fan speed) (rpm)',
  '(Corrected core speed) (rpm)',
  '(Bypass Ratio) ',
  '(Bleed Enthalpy)',
  '(High-pressure turbines Cool air flow)',
  '(Low-pressure turbines Cool air flow)']]

In [14]:
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer, accuracy_score

import sklearn
from sklearn.metrics import mean_squared_error, r2_score


In [15]:
feature_names = Selected_Features[0]
np.shape(X)

(20631, 18)

## **Final Analysis of Cycle Data**

In [16]:
# ============================================================
# Prepare Test Dataset for Final Engine Cycle Analysis
# ============================================================

# Find the maximum cycle value for each engine
# This represents the last recorded operational cycle
df_test_cycle = df_test.groupby(['engine']).agg({
    'cycle': 'max'
})

# Rename the column for better readability
df_test_cycle.rename(
    columns={'cycle': 'life'},
    inplace=True
)


# ============================================================
# Merge Maximum Cycle Information with Test Dataset
# ============================================================

# Add the maximum cycle ('life') information
# to each row in the test dataset
df_test_max = df_test.merge(
    df_test_cycle,
    how='left',
    on=['engine']
)


# ============================================================
# Keep Only Final Cycle Records
# ============================================================

# Select only rows where:
# current cycle == maximum cycle
#
# These rows represent the latest condition
# of each engine before prediction
df_test_max = df_test_max[
    df_test_max['cycle'] == df_test_max['life']
]


# ============================================================
# Remove Temporary Columns
# ============================================================

# 'life' column is no longer needed
df_test_max.drop(
    ['life'],
    axis=1,
    inplace=True
)

# Preview dataset (optional)
# df_test_max

## **Training, Test and Learning**

In [17]:
X_train = df_train[feature_names]
y_train = df_train.iloc[:,-1]
X_test = df_test_max[feature_names]
y_test = df_test_RUL.iloc[:,-1]

In [18]:
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [19]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

## **KNN Regression**

In [20]:
# ============================================================
# Train and Evaluate K-Nearest Neighbors (KNN) Regressor
# ============================================================

# Measure total execution time of the cell
%%time

# Import required model
from sklearn.neighbors import KNeighborsRegressor

# ============================================================
# Record Training Start Time
# ============================================================

start = time.time()


# ============================================================
# Build and Train the KNN Regression Model
# ============================================================

# Create KNN regressor with 9 nearest neighbors
model = KNeighborsRegressor(
    n_neighbors=9
)

# Train the model using training data
model.fit(X_train, y_train)

# Record training completion time
end_train = time.time()


# ============================================================
# Generate Predictions on Test Data
# ============================================================

# Predict Remaining Useful Life (RUL) values
y_predictions = model.predict(X_test)

# Record prediction completion time
end_predict = time.time()


# ============================================================
# Model Evaluation
# ============================================================

# Calculate R² Score
# Measures how well the model explains variance in the data
r2 = model.score(X_test, y_test)

# Calculate Root Mean Squared Error (RMSE)
# Lower RMSE indicates better prediction accuracy
rmse = mean_squared_error(
    y_test,
    y_predictions,
    squared=False
)


# ============================================================
# Display Evaluation Results
# ============================================================

print('R-squared error: ' + "{:.2%}".format(r2))

print('Root Mean Squared Error: ' + "{:.2f}".format(rmse))

R-squared error: 80.17%
Root Mean Squared Error: 18.50
CPU times: user 103 ms, sys: 13.4 ms, total: 116 ms
Wall time: 52.7 ms


## **SVR**

In [21]:
# ============================================================
# Train and Evaluate Support Vector Regression (SVR) Model
# ============================================================

# Measure total execution time of the cell
%%time

# Import Support Vector Regression model
from sklearn.svm import SVR


# ============================================================
# Record Training Start Time
# ============================================================

start = time.time()


# ============================================================
# Build and Train the SVR Model
# ============================================================

# Create SVR model with RBF (Radial Basis Function) kernel
#
# Parameters:
#   kernel  -> Defines the kernel type
#   C       -> Regularization parameter
#   gamma   -> Controls influence of training samples
#   epsilon -> Margin of tolerance for error
model = SVR(
    kernel="rbf",
    C=100,
    gamma=0.5,
    epsilon=0.01
)

# Train the model using training data
model.fit(X_train, y_train)

# Record training completion time
end_train = time.time()


# ============================================================
# Generate Predictions on Test Data
# ============================================================

# Predict Remaining Useful Life (RUL) values
y_predictions = model.predict(X_test)

# Record prediction completion time
end_predict = time.time()


# ============================================================
# Model Evaluation
# ============================================================

# Calculate R² Score
# Indicates how well predictions fit the actual values
r2 = model.score(X_test, y_test)

# Calculate Root Mean Squared Error (RMSE)
# Lower RMSE indicates better predictive performance
rmse = mean_squared_error(
    y_test,
    y_predictions,
    squared=False
)


# ============================================================
# Display Evaluation Results
# ============================================================

print('R-squared error: ' + "{:.2%}".format(r2))

print('Root Mean Squared Error: ' + "{:.2f}".format(rmse))

R-squared error: 79.10%
Root Mean Squared Error: 19.00
CPU times: user 18.7 s, sys: 158 ms, total: 18.8 s
Wall time: 18.8 s


## **Random Forest Regressor**

In [22]:
# ============================================================
# Train and Evaluate Random Forest Regressor
# ============================================================

# Measure total execution time of the cell
%%time

# Import Random Forest Regression model
from sklearn.ensemble import RandomForestRegressor


# ============================================================
# Record Training Start Time
# ============================================================

start = time.time()


# ============================================================
# Build and Train the Random Forest Model
# ============================================================

# Create Random Forest Regressor
#
# Parameters:
#   n_jobs           -> Use all available CPU cores for parallel processing
#   n_estimators     -> Number of decision trees in the forest
#   min_samples_leaf -> Minimum number of samples required at leaf nodes
#   max_features     -> Number of features considered at each split
model = RandomForestRegressor(
    n_jobs=-1,
    n_estimators=500,
    min_samples_leaf=1,
    max_features='sqrt'
)

# Train the model using training data
model.fit(X_train, y_train)

# Record training completion time
end_train = time.time()


# ============================================================
# Generate Predictions on Test Data
# ============================================================

# Predict Remaining Useful Life (RUL) values
y_predictions = model.predict(X_test)

# Record prediction completion time
end_predict = time.time()


# ============================================================
# Model Evaluation
# ============================================================

# Calculate R² Score
# Measures how well the model explains variance in target values
r2 = model.score(X_test, y_test)

# Calculate Root Mean Squared Error (RMSE)
# Lower RMSE indicates better model performance
rmse = mean_squared_error(
    y_test,
    y_predictions,
    squared=False
)


# ============================================================
# Display Evaluation Results
# ============================================================

print('R-squared error: ' + "{:.2%}".format(r2))

print('Root Mean Squared Error: ' + "{:.2f}".format(rmse))

R-squared error: 81.15%
Root Mean Squared Error: 18.04
CPU times: user 39.2 s, sys: 553 ms, total: 39.8 s
Wall time: 10.3 s


## **Let's go to the neural network - LSTM**

In [23]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense, LSTM, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import sklearn

2025-09-15 17:19:45.167760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757956785.363943      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757956785.426110      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [24]:
print(X_train.shape)

(20631, 16)


## **train/test**

In [25]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test  = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train_3D shape:", X_train.shape)
print("X_test_3D shape:", X_test.shape)

X_train_3D shape: (20631, 1, 16)
X_test_3D shape: (100, 1, 16)


### **training**

In [26]:
# ============================================================
# Import Required TensorFlow / Keras Modules
# ============================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# Build LSTM-Based Deep Learning Model
# ============================================================

# Create Sequential model
model = Sequential()


# ============================================================
# First LSTM Layer
# ============================================================

# Parameters:
#   128               -> Number of memory units
#   return_sequences  -> Required because another LSTM layer follows
#   input_shape       -> Shape of input time-series data
model.add(
    LSTM(
        128,
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2])
    )
)

# Normalize activations for more stable training
model.add(BatchNormalization())

# Reduce overfitting
model.add(Dropout(0.3))


# ============================================================
# Second LSTM Layer
# ============================================================

# Additional temporal feature extraction
model.add(
    LSTM(
        64,
        return_sequences=True,
        activation='tanh'
    )
)

# Dropout regularization
model.add(Dropout(0.3))


# ============================================================
# Third LSTM Layer
# ============================================================

# Final LSTM layer
# return_sequences=False because this is the last recurrent layer
model.add(
    LSTM(
        32,
        activation='tanh'
    )
)

# Dropout regularization
model.add(Dropout(0.3))


# ============================================================
# Fully Connected (Dense) Layers
# ============================================================

# Hidden dense layer for nonlinear learning
model.add(
    Dense(
        64,
        activation='relu'
    )
)

# Output layer
# Single neuron for regression output (RUL prediction)
model.add(Dense(1))


# ============================================================
# Compile the Model
# ============================================================

# optimizer -> Adam optimization algorithm
# loss      -> Mean Squared Error for regression
# metrics   -> Mean Absolute Error for monitoring
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)


# ============================================================
# Configure Early Stopping
# ============================================================

# Stops training if validation loss does not improve
# for 10 consecutive epochs
#
# restore_best_weights=True:
# Restores weights from the best validation performance
es = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


# ============================================================
# Train the Model
# ============================================================

history = model.fit(
    X_train,
    y_train,

    # Validation dataset
    validation_data=(X_test, y_test),

    # Maximum training epochs
    epochs=100,

    # Number of samples processed before updating weights
    batch_size=64,

    # Apply early stopping callback
    callbacks=[es]
)

I0000 00:00:1757956797.474057      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1757956797.474716      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/100


I0000 00:00:1757956804.135482      92 cuda_dnn.cc:529] Loaded cuDNN version 90300


323/323 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6093.7183 - mae: 66.3004 - val_loss: 595.9051 - val_mae: 20.5107
Epoch 2/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 529.7598 - mae: 18.0944 - val_loss: 397.9867 - val_mae: 14.2489
Epoch 3/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 419.7045 - mae: 15.5560 - val_loss: 358.3123 - val_mae: 13.9600
Epoch 4/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 400.9113 - mae: 15.2440 - val_loss: 341.4629 - val_mae: 13.1885
Epoch 5/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 395.4748 - mae: 14.9833 - val_loss: 330.7992 - val_mae: 13.2024
Epoch 6/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 381.9922 - mae: 14.8401 - val_loss: 345.7232 - val_mae: 13.5805
Epoch 7/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 375.8253 - mae: 14.6995 - val_loss: 332.6303 - val_mae: 13.3767
Epoch 8/100
323/323 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 362.2030 - mae: 14.3536 - val_loss: 316.5907 - val_mae: 13.2786
Epoch 9/1

## **Prediction & Evalution**

In [27]:
# ============================================================
# Import Required Libraries
# ============================================================

import numpy as np

# Model evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)


# ============================================================
# Generate Predictions
# ============================================================

# Predict Remaining Useful Life (RUL)
# using the trained model
y_pred = model.predict(X_test)


# ============================================================
# Evaluate Model Performance
# ============================================================

# Mean Absolute Error (MAE)
# Measures the average absolute prediction error
mae = mean_absolute_error(y_test, y_pred)

# R² Score
# Indicates how well predictions fit the true values
r2 = r2_score(y_test, y_pred)

# Convert R² score to percentage format
accuracy_percent = r2 * 100


# ============================================================
# Display Evaluation Results
# ============================================================

print("Accuracy (%):", accuracy_percent)

print("MAE:", mae)

print("R2 Score:", r2)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step
Accuracy (%): 81.666783740912
MAE: 13.27859601020813
R2 Score: 0.81666783740912
